In [86]:
import pandas as pd
import numpy as np

# Load datasets
df_2009 = pd.read_csv("2009-2020.csv")
df_2024 = pd.read_csv("2024.csv")
df_2025 = pd.read_csv("2025.csv")

In [87]:
df_2009 = df_2009[["date","headline"]]

In [88]:
df_2024 = df_2024[["Date","CompactedSummary"]]

In [89]:
df_2025 = df_2025[["Date","Headline"]]

In [90]:
df_2024.rename(columns={"Date":"date","CompactedSummary":"headline"},inplace = True)
df_2025.rename(columns={"Date":"date","Headline":"headline"},inplace = True)

In [91]:
def clean_dates(df):
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    return df

In [92]:
df_2009 = clean_dates(df_2009)
df_2024 = clean_dates(df_2024)
df_2025 = clean_dates(df_2025)

C:\Users\USER\AppData\Local\Temp\ipykernel_3692\2654819562.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")


In [93]:
df_2009.dtypes

date        datetime64[us]
headline               str
dtype: object

In [94]:
df_2009.dropna(inplace=True)
df_2024.dropna(inplace=True)
df_2025.dropna(inplace=True)

In [106]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()

def fetch_news_for_day(date):
    url = "https://newsapi.org/v2/everything"

    params = {
        "q": "Microsoft OR MSFT",
        "from": date,
        "to": date,
        "language": "en",
        "sortBy": "relevancy",
        "apiKey": os.getenv("NEWS_API")
    }

    r = requests.get(url, params=params)
    data = r.json()

    if data.get("status") != "ok":
        return []

    articles = data.get("articles", [])

    rows = []
    for article in articles:
        rows.append({
            "date": article["publishedAt"].split("T")[0],
            "headline": article["title"],
        })

    return rows

In [107]:
from datetime import datetime, timedelta

start_date = datetime(2026, 3, 19)
end_date = datetime(2026, 4, 20)

all_data = []

current = start_date

while current <= end_date:
    date_str = current.strftime("%Y-%m-%d")
    print("Fetching:", date_str)

    try:
        daily_news = fetch_news_for_day(date_str)
        all_data.extend(daily_news)

    except Exception as e:
        print("Error on", date_str, e)

    current += timedelta(days=1)

Fetching: 2026-03-19
Fetching: 2026-03-20
Fetching: 2026-03-21
Fetching: 2026-03-22
Fetching: 2026-03-23
Fetching: 2026-03-24
Fetching: 2026-03-25
Fetching: 2026-03-26
Fetching: 2026-03-27
Fetching: 2026-03-28
Fetching: 2026-03-29
Fetching: 2026-03-30
Fetching: 2026-03-31
Fetching: 2026-04-01
Fetching: 2026-04-02
Fetching: 2026-04-03
Fetching: 2026-04-04
Fetching: 2026-04-05
Fetching: 2026-04-06
Fetching: 2026-04-07
Fetching: 2026-04-08
Fetching: 2026-04-09
Fetching: 2026-04-10
Fetching: 2026-04-11
Fetching: 2026-04-12
Fetching: 2026-04-13
Fetching: 2026-04-14
Fetching: 2026-04-15
Fetching: 2026-04-16
Fetching: 2026-04-17
Fetching: 2026-04-18
Fetching: 2026-04-19
Fetching: 2026-04-20


In [108]:
df = pd.DataFrame(all_data)
df = df.dropna()
df = df.drop_duplicates()
df = clean_dates(df)
# news_df.to_csv("msft_news_2026.csv", index=False)

In [109]:
news_df = pd.concat([df_2009, df_2024, df_2025,df], ignore_index=True)

In [110]:
news_df

,date,headline
0,2020-06-01,Agilent Technologies Announces Pricing of $5……...
1,2020-05-18,Agilent (A) Gears Up for Q2 Earnings: What's i...
2,2020-05-15,J.P. Morgan Asset Management Announces Liquida...
3,2020-05-15,"Pershing Square Capital Management, L.P. Buys ..."
4,2020-05-12,Agilent Awards Trilogy Sciences with a Golden ...
...,...,...
1853245,2026-04-18,Curious About Claude? Try These 8 Beginner-Fri...
1853246,2026-04-18,Razer Pro Type Ergo
1853247,2026-04-18,Three more senior executives leave OpenAI as t...
1853248,2026-04-18,Anthropic’s Amodei meets Wiles and Bessent at ...


In [111]:
news_df = news_df.sort_values("date").reset_index(drop=True)

In [112]:
# Filter Microsoft-related news
keywords = ["microsoft", "msft"]

def is_msft(text):
    if pd.isna(text):
        return False
    text = text.lower()
    return any(k in text for k in keywords)

news_df = news_df[news_df['headline'].apply(is_msft)]

In [113]:
news_df

,date,headline
369,2010-03-01,Google Fears Microsoft's Cloaks & Daggers: Tod...
839,2010-03-08,Microsoft Back-Up Made Simple by Rebit
1087,2010-03-11,Forbes' Richest: Mobile Beats Microsoft
1284,2010-03-16,Microsoft Clinches Cisco Server Deal
2821,2010-04-15,Microsoft NERDs Talk Up Military Robots
...,...,...
1853236,2026-04-18,vox-microsoft 0.2.8
1853237,2026-04-18,vox-microsoft 0.2.11
1853238,2026-04-18,vox-microsoft 0.2.14
1853245,2026-04-18,Turn your office suite around with this deal o...


In [114]:
news_df.drop_duplicates(inplace=True)

In [115]:
news_df.to_csv("news_dataset.csv", index=False)

In [182]:
news_df.dtypes

date         datetime64[us]
headline                str
sentiment             int64
dtype: object

In [165]:
import yfinance as yf

data = yf.download("MSFT", start="2010-01-01")

# print(data)

data.to_csv("msft.csv")

[*********************100%***********************]  1 of 1 completed


In [166]:
ohlcv_df = pd.read_csv('msft.csv')

In [167]:
ohlcv_df = ohlcv_df.dropna()
ohlcv_df = ohlcv_df.drop_duplicates()

In [168]:
# reset index so Date becomes a column
ohlcv_df = ohlcv_df.drop(0)   # removes row with index 0,1
ohlcv_df = ohlcv_df.reset_index(drop=True)
ohlcv_df = ohlcv_df.rename(columns={"Price": "date",})
# rename columns
ohlcv_df.head()

,date,Close,High,Low,Open,Volume
0,2010-01-04,23.077375411987305,23.189220239875745,22.808946687307735,22.83131622175908,38409100
1,2010-01-05,23.084835052490234,23.189224811659603,22.846232326704975,23.002816254366817,49749600
2,2010-01-06,22.94316291809082,23.174309104175542,22.756754382367983,23.02518170672357,58182400
3,2010-01-07,22.704570770263672,22.890979392606774,22.510705632364658,22.83878378371537,50559700
4,2010-01-08,22.86114501953125,23.0251840355725,22.54797859095475,22.577804642335565,51197400


In [169]:
ohlcv_df.dtypes

date      str
Close     str
High      str
Low       str
Open      str
Volume    str
dtype: object

In [171]:
cols = ohlcv_df.columns.drop(["date"])
ohlcv_df[cols] = ohlcv_df[cols].astype(float)

In [172]:
import datetime

def str_to_datetime(s):
  split = s.split('-')
  year, month, day = int(split[0]), int(split[1]), int(split[2])
  return datetime.datetime(year=year, month=month, day=day)

datetime_object = str_to_datetime('1986-03-19')
datetime_object

datetime.datetime(1986, 3, 19, 0, 0)

In [173]:
ohlcv_df['date'] = ohlcv_df['date'].apply(str_to_datetime)
ohlcv_df['date']

0      2010-01-04
1      2010-01-05
2      2010-01-06
3      2010-01-07
4      2010-01-08
          ...    
4092   2026-04-13
4093   2026-04-14
4094   2026-04-15
4095   2026-04-16
4096   2026-04-17
Name: date, Length: 4097, dtype: datetime64[us]

In [125]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch

model_name = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

Loading weights: 100%|████████████████████████████| 201/201 [00:00<00:00, 7058.63it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
label_map = {
    "positive": 1,
    "negative": -1,
    "neutral": 0
}

def get_sentiment(text):
    try:
        result = sentiment_pipeline(str(text[:512]))[0]
        label = result['label'].lower()
        return label_map.get(label, 0)
    except:
        return 0

In [127]:
from tqdm import tqdm
tqdm.pandas()

news_df["sentiment"] = news_df["headline"].progress_apply(get_sentiment)

100%|█████████████████████████████████████████████| 2727/2727 [03:21<00:00, 13.55it/s]


In [128]:
news_df

,date,headline,sentiment
369,2010-03-01,Google Fears Microsoft's Cloaks & Daggers: Tod...,0
839,2010-03-08,Microsoft Back-Up Made Simple by Rebit,0
1087,2010-03-11,Forbes' Richest: Mobile Beats Microsoft,0
1284,2010-03-16,Microsoft Clinches Cisco Server Deal,0
2821,2010-04-15,Microsoft NERDs Talk Up Military Robots,0
...,...,...,...
1853236,2026-04-18,vox-microsoft 0.2.8,0
1853237,2026-04-18,vox-microsoft 0.2.11,0
1853238,2026-04-18,vox-microsoft 0.2.14,0
1853245,2026-04-18,Turn your office suite around with this deal o...,0


In [162]:
daily_sentiment = news_df.groupby("date").agg(
    sentiment_mean=("sentiment", "mean"),
    # news_count=("sentiment", "count")
).reset_index()

# Normalize sentiment
daily_sentiment["sentiment_mean"] = daily_sentiment["sentiment_mean"].fillna(0)

In [163]:
daily_sentiment

,date,sentiment_mean
0,2010-03-01,0.000000
1,2010-03-08,0.000000
2,2010-03-11,0.000000
3,2010-03-16,0.000000
4,2010-04-15,0.000000
...,...,...
1122,2026-04-14,-0.177778
1123,2026-04-15,-0.194444
1124,2026-04-16,0.000000
1125,2026-04-17,0.024390


In [174]:
final_df = pd.merge(
    ohlcv_df,
    daily_sentiment,
    on="date",
    how="left"
)

# Fill missing sentiment
final_df["sentiment_mean"] = final_df["sentiment_mean"].fillna(0)

In [175]:
final_df

,date,Close,High,Low,Open,Volume,sentiment_mean
0,2010-01-04,23.077375,23.189220,22.808947,22.831316,38409100.0,0.000000
1,2010-01-05,23.084835,23.189225,22.846232,23.002816,49749600.0,0.000000
2,2010-01-06,22.943163,23.174309,22.756754,23.025182,58182400.0,0.000000
3,2010-01-07,22.704571,22.890979,22.510706,22.838784,50559700.0,0.000000
4,2010-01-08,22.861145,23.025184,22.547979,22.577805,51197400.0,0.000000
...,...,...,...,...,...,...,...
4092,2026-04-13,384.369995,384.540009,371.019989,373.609985,35745800.0,-0.238095
4093,2026-04-14,393.109985,394.690002,386.519989,387.920013,37504500.0,-0.177778
4094,2026-04-15,411.220001,414.369995,396.730011,398.000000,45063400.0,-0.194444
4095,2026-04-16,420.260010,420.820007,412.140015,419.859985,41642400.0,0.000000


In [179]:
final_df["sentiment_3d"] = final_df["sentiment_mean"].rolling(3).mean().fillna(0)
final_df["sentiment_7d"] = final_df["sentiment_mean"].rolling(7).mean().fillna(0)
final_df["sentiment_14d"] = final_df["sentiment_mean"].rolling(14).mean().fillna(0)

In [183]:
final_df.dtypes

date              datetime64[us]
Close                    float64
High                     float64
Low                      float64
Open                     float64
Volume                   float64
sentiment_mean           float64
sentiment_3d             float64
sentiment_7d             float64
sentiment_14d            float64
dtype: object

In [180]:
final_df.to_csv("msft_dataset.csv", index=False)